In [1]:
# THE AI POWER MAP
# Entrega 2 — Perfilado, Diccionario y Limpieza Inicial
# Equipo: DataNova Lab

!pip install pandas matplotlib seaborn missingno -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print(" Librerías cargadas correctamente")

 Librerías cargadas correctamente


In [3]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Ciclo 8/Data Visualizacion/Data_Visualizacion_TF/Fuentes en csv/all_ai_models.csv')

print(f" Dataset cargado")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head(3)

ModuleNotFoundError: No module named 'google'

In [ ]:
# Dimensiones y tipos de datos
print("="*60)
print("INFORMACIÓN GENERAL DEL DATASET")
print("="*60)
print(f"\nFilas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"\nTipos de datos:")
print(df.dtypes)

INFORMACIÓN GENERAL DEL DATASET

Filas: 3509
Columnas: 57

Tipos de datos:
Model                                  object
Domain                                 object
Task                                   object
Organization                           object
Authors                                object
Publication date                       object
Reference                              object
Link                                   object
Citations                             float64
Notability criteria                    object
Notability criteria notes              object
Parameters                            float64
Parameters notes                       object
Training compute (FLOP)               float64
Training compute notes                 object
Training dataset size (total)          object
Dataset size notes                     object
Training time (hours)                 float64
Training time notes                    object
Training hardware                      object
Appro

In [ ]:
print("="*60)
print("ANÁLISIS DE VALORES NULOS")
print("="*60)

nulos = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Porcentaje (%)', ascending=False)

print(nulos.to_string())

ANÁLISIS DE VALORES NULOS
                                    Nulos  Porcentaje (%)
Post-training compute notes          3508           99.97
Post-training compute (FLOP)         3508           99.97
Archived links                       3484           99.29
Hardware utilization (HFU)           3484           99.29
Training compute lower bound         3483           99.26
Training compute upper bound         3468           98.83
Training cloud compute vendor        3456           98.49
Hardware utilization (MFU)           3448           98.26
Training data center                 3443           98.12
Foundation model                     3431           97.78
Utilization notes                    3421           97.49
Frontier model                       3372           96.10
Batch size notes                     3312           94.39
Training chip-hours                  3294           93.87
Training compute cost (2023 USD)     3285           93.62
Batch size                           3258     

#Eliminamos a los nulos que son el 90% o mas.

In [ ]:
# Columnas que conservamos para el análisis
columnas_utiles = [
    # Identificación
    'Model',
    'Organization',
    'Country (of organization)',
    'Publication date',
    'Organization categorization',

    # Clasificación
    'Domain',
    'Task',

    # Escala técnica
    'Parameters',
    'Training compute (FLOP)',
    'Training dataset size (total)',
    'Training time (hours)',

    # Costo y hardware
    'Training compute cost (2023 USD)',
    'Hardware quantity',
    'Training hardware',

    # Accesibilidad
    'Model accessibility',
    'Training code accessibility',
    'Open model weights?',

    # Referencias
    'Citations',
    'Confidence',
    'Notability criteria',
]

df_clean = df[columnas_utiles].copy()

print(f"Columnas originales: {df.shape[1]}")
print(f"Columnas seleccionadas: {df_clean.shape[1]}")
print(f"Columnas eliminadas: {df.shape[1] - df_clean.shape[1]}")
print(f"\nColumnas conservadas:")
for col in df_clean.columns:
    pct = df_clean[col].isnull().mean() * 100
    print(f"  {col}: {pct:.1f}% nulos")

Columnas originales: 57
Columnas seleccionadas: 20
Columnas eliminadas: 37

Columnas conservadas:
  Model: 0.0% nulos
  Organization: 2.5% nulos
  Country (of organization): 2.7% nulos
  Publication date: 0.7% nulos
  Organization categorization: 3.0% nulos
  Domain: 2.7% nulos
  Task: 3.5% nulos
  Parameters: 34.8% nulos
  Training compute (FLOP): 60.4% nulos
  Training dataset size (total): 59.4% nulos
  Training time (hours): 84.3% nulos
  Training compute cost (2023 USD): 93.6% nulos
  Hardware quantity: 76.0% nulos
  Training hardware: 66.4% nulos
  Model accessibility: 24.7% nulos
  Training code accessibility: 31.7% nulos
  Open model weights?: 24.7% nulos
  Citations: 58.3% nulos
  Confidence: 5.6% nulos
  Notability criteria: 70.6% nulos


In [ ]:
# Publication date está como object — la convertimos a datetime
df_clean['Publication date'] = pd.to_datetime(df_clean['Publication date'], errors='coerce')

# Extraemos año y mes para análisis temporal
df_clean['Year'] = df_clean['Publication date'].dt.year
df_clean['Month'] = df_clean['Publication date'].dt.month

# Training dataset size está como object — la convertimos a numérica
df_clean['Training dataset size (total)'] = pd.to_numeric(
    df_clean['Training dataset size (total)'], errors='coerce'
)

print("Tipos de datos corregidos:")
print(df_clean.dtypes)
print(f"\nRango temporal del dataset:")
print(f"  Desde: {df_clean['Year'].min()}")
print(f"  Hasta: {df_clean['Year'].max()}")
print(f"  Modelos sin fecha: {df_clean['Year'].isnull().sum()}")

Tipos de datos corregidos:
Model                                       object
Organization                                object
Country (of organization)                   object
Publication date                    datetime64[ns]
Organization categorization                 object
Domain                                      object
Task                                        object
Parameters                                 float64
Training compute (FLOP)                    float64
Training dataset size (total)              float64
Training time (hours)                      float64
Training compute cost (2023 USD)           float64
Hardware quantity                          float64
Training hardware                           object
Model accessibility                         object
Training code accessibility                 object
Open model weights?                         object
Citations                                  float64
Confidence                                  object
Nota

In [ ]:
print("="*60)
print("ANÁLISIS DE DUPLICADOS")
print("="*60)

# Duplicados exactos
duplicados_exactos = df_clean.duplicated().sum()
print(f"\nDuplicados exactos: {duplicados_exactos}")

# Duplicados por nombre de modelo
duplicados_modelo = df_clean.duplicated(subset=['Model']).sum()
print(f"Modelos con nombre duplicado: {duplicados_modelo}")

# Ver cuáles son
if duplicados_modelo > 0:
    duplicados_detalle = df_clean[df_clean.duplicated(subset=['Model'], keep=False)]
    print(f"\nEjemplos de modelos duplicados:")
    print(duplicados_detalle[['Model', 'Organization', 'Year', 'Domain']].head(10).to_string())

ANÁLISIS DE DUPLICADOS

Duplicados exactos: 0
Modelos con nombre duplicado: 6

Ejemplos de modelos duplicados:
                Model                                                                                                                                                               Organization    Year                      Domain
13     Gemini 3.1 Pro                                                                                                                                                            Google DeepMind 2026.00  Multimodal,Language,Vision
14     Gemini 3.1 Pro                                                                                                                                                            Google DeepMind 2026.00             Language,Vision
17              GLM-5                                                                                                                                                            Z.ai (Zhipu AI) 2026.00  

In [ ]:
# Nos quedamos con el primer registro de cada modelo
# que es el que tiene más datos completos en Epoch AI (Osea nos quedaremos con el mas actualizado)
df_clean = df_clean.drop_duplicates(subset=['Model'], keep='first')

print(f"Filas antes: 3509")
print(f"Filas después: {df_clean.shape[0]}")
print(f"Duplicados eliminados: {3509 - df_clean.shape[0]}")

Filas antes: 3509
Filas después: 3503
Duplicados eliminados: 6


In [ ]:
print("="*60)
print("CARDINALIDAD DE VARIABLES CATEGÓRICAS")
print("="*60)

categoricas = ['Domain', 'Task', 'Organization categorization',
               'Model accessibility', 'Training code accessibility',
               'Open model weights?', 'Confidence', 'Country (of organization)']

for col in categoricas:
    n = df_clean[col].nunique()
    top5 = df_clean[col].value_counts().head(5)
    print(f"\n{col} — {n} valores únicos:")
    print(top5.to_string())

CARDINALIDAD DE VARIABLES CATEGÓRICAS

Domain — 173 valores únicos:
Domain
Language            1548
Biology              376
Vision               322
Image generation     163
Speech               129

Task — 919 valores únicos:
Task
Language modeling                                  317
Language modeling/generation                       178
Language modeling/generation,Question answering    156
Image classification                               127
Image generation,Text-to-image                      67

Organization categorization — 126 valores únicos:
Organization categorization
Industry             1737
Academia              529
Academia,Academia     187
Industry,Academia     159
Academia,Industry     123

Model accessibility — 6 valores únicos:
Model accessibility
Unreleased                       828
Open weights (unrestricted)      797
API access                       369
Open weights (restricted use)    296
Open weights (non-commercial)    225

Training code accessibility — 4 valo

#Analisis de valores unicos

In [ ]:
print("="*60)
print("CARDINALIDAD DE VARIABLES CATEGÓRICAS")
print("="*60)

categoricas = ['Domain', 'Task', 'Organization categorization',
               'Model accessibility', 'Training code accessibility',
               'Open model weights?', 'Confidence', 'Country (of organization)']

for col in categoricas:
    n = df_clean[col].nunique()
    top5 = df_clean[col].value_counts().head(5)
    print(f"\n{col} — {n} valores únicos:")
    print(top5.to_string())

CARDINALIDAD DE VARIABLES CATEGÓRICAS

Domain — 173 valores únicos:
Domain
Language            1548
Biology              376
Vision               322
Image generation     163
Speech               129

Task — 919 valores únicos:
Task
Language modeling                                  317
Language modeling/generation                       178
Language modeling/generation,Question answering    156
Image classification                               127
Image generation,Text-to-image                      67

Organization categorization — 126 valores únicos:
Organization categorization
Industry             1737
Academia              529
Academia,Academia     187
Industry,Academia     159
Academia,Industry     123

Model accessibility — 6 valores únicos:
Model accessibility
Unreleased                       828
Open weights (unrestricted)      797
API access                       369
Open weights (restricted use)    296
Open weights (non-commercial)    225

Training code accessibility — 4 valo

In [ ]:
# ── 1. Limpiar Country: quedarnos con el primer país cuando hay combinaciones
df_clean['Country (of organization)'] = (
    df_clean['Country (of organization)']
    .str.split(',')
    .str[0]
    .str.strip()
)

# ── 2. Limpiar Organization categorization: quedarnos con el primero
df_clean['Organization categorization'] = (
    df_clean['Organization categorization']
    .str.split(',')
    .str[0]
    .str.strip()
)

# ── 3. Simplificar Domain: quedarnos con el dominio principal
df_clean['Domain'] = (
    df_clean['Domain']
    .str.split(',')
    .str[0]
    .str.strip()
)

# ── Verificar resultados
print("Country — valores únicos después de limpieza:")
print(df_clean['Country (of organization)'].value_counts().head(10).to_string())

print("\nOrganization categorization — valores únicos:")
print(df_clean['Organization categorization'].value_counts().to_string())

print("\nDomain — valores únicos:")
print(df_clean['Domain'].value_counts().head(10).to_string())

Country — valores únicos después de limpieza:
Country (of organization)
United States of America                                1702
China                                                    809
United Kingdom of Great Britain and Northern Ireland     179
Canada                                                   105
Korea (Republic of)                                       77
Germany                                                   74
France                                                    72
Japan                                                     53
Switzerland                                               49
Hong Kong                                                 38

Organization categorization — valores únicos:
Organization categorization
Industry               2125
Academia               1174
Research collective      68
Government               31

Domain — valores únicos:
Domain
Language            1668
Vision               399
Biology              384
Multimodal           19

In [ ]:
# Eliminamos filas sin país, sin organización y sin fecha
# porque son las dimensiones principales del análisis

antes = df_clean.shape[0]

df_clean = df_clean.dropna(subset=[
    'Country (of organization)',
    'Organization',
    'Publication date'
])

despues = df_clean.shape[0]

print(f"Filas antes: {antes}")
print(f"Filas después: {despues}")
print(f"Filas eliminadas: {antes - despues}")
print(f"\nNulos restantes por columna:")
nulos_restantes = df_clean.isnull().sum()
nulos_restantes = nulos_restantes[nulos_restantes > 0]
print(nulos_restantes.to_string())

Filas antes: 3503
Filas después: 3405
Filas eliminadas: 98

Nulos restantes por columna:
Organization categorization           16
Domain                                40
Task                                  66
Parameters                          1162
Training compute (FLOP)             2039
Training dataset size (total)       2026
Training time (hours)               2861
Training compute cost (2023 USD)    3181
Hardware quantity                   2567
Training hardware                   2235
Model accessibility                  797
Training code accessibility         1024
Open model weights?                  797
Citations                           1943
Confidence                           197
Notability criteria                 2395


In [ ]:
# Las columnas categóricas con pocos nulos las imputamos con 'Unknown'
# para no perder filas en el análisis

df_clean['Organization categorization'] = df_clean['Organization categorization'].fillna('Unknown')
df_clean['Domain'] = df_clean['Domain'].fillna('Unknown')
df_clean['Task'] = df_clean['Task'].fillna('Unknown')
df_clean['Model accessibility'] = df_clean['Model accessibility'].fillna('Unknown')
df_clean['Training code accessibility'] = df_clean['Training code accessibility'].fillna('Unknown')
df_clean['Open model weights?'] = df_clean['Open model weights?'].fillna('Unknown')
df_clean['Confidence'] = df_clean['Confidence'].fillna('Unknown')
df_clean['Notability criteria'] = df_clean['Notability criteria'].fillna('Unknown')

print("Nulos restantes en categóricas:")
categoricas = ['Organization categorization', 'Domain', 'Task',
               'Model accessibility', 'Training code accessibility',
               'Open model weights?', 'Confidence', 'Notability criteria']

for col in categoricas:
    print(f"  {col}: {df_clean[col].isnull().sum()}")

print(f"\nTotal filas: {df_clean.shape[0]}")

Nulos restantes en categóricas:
  Organization categorization: 0
  Domain: 0
  Task: 0
  Model accessibility: 0
  Training code accessibility: 0
  Open model weights?: 0
  Confidence: 0
  Notability criteria: 0

Total filas: 3405


# VERIFICACION 1 DE DATA SET LIMPIO, Guardamos el data set

In [ ]:
# Guardamos el dataset limpio listo para Tableau y siguientes notebooks
df_clean.to_csv('all_ai_models_clean.csv', index=False)

print("="*60)
print("RESUMEN FINAL DEL PERFILADO Y LIMPIEZA")
print("="*60)
print(f"\nDataset original:       3,509 filas | 57 columnas")
print(f"Dataset limpio:         {df_clean.shape[0]} filas | {df_clean.shape[1]} columnas")
print(f"\nColumnas eliminadas:    37 (más del 90% de nulos o irrelevantes)")
print(f"Filas eliminadas:       104 (sin país, organización o fecha)")
print(f"Duplicados eliminados:  6")
print(f"\nPeriodo cubierto:       {int(df_clean['Year'].min())} – {int(df_clean['Year'].max())}")
print(f"Países únicos:          {df_clean['Country (of organization)'].nunique()}")
print(f"Organizaciones únicas:  {df_clean['Organization'].nunique()}")
print(f"Dominios únicos:        {df_clean['Domain'].nunique()}")
print(f"\n Dataset exportado como: all_ai_models_clean.csv")

RESUMEN FINAL DEL PERFILADO Y LIMPIEZA

Dataset original:       3,509 filas | 57 columnas
Dataset limpio:         3405 filas | 22 columnas

Columnas eliminadas:    37 (más del 90% de nulos o irrelevantes)
Filas eliminadas:       104 (sin país, organización o fecha)
Duplicados eliminados:  6

Periodo cubierto:       1950 – 2026
Países únicos:          45
Organizaciones únicas:  1306
Dominios únicos:        22

 Dataset exportado como: all_ai_models_clean.csv


#VERIFICACION 2 DE DATA SET LIMPIO. SOLO SE VERIFICA - REGLAS, No hace nda.

In [ ]:
# ============================================================
# LIMPIEZA COMPLETA — 7 reglas
# ============================================================

df_clean = df.copy()

# REGLA 1 — Selección de columnas útiles (eliminamos las de +90% nulos)
columnas_utiles = [
    'Model', 'Organization', 'Country (of organization)',
    'Publication date', 'Organization categorization',
    'Domain', 'Task', 'Parameters',
    'Training compute (FLOP)', 'Training dataset size (total)',
    'Training time (hours)', 'Training compute cost (2023 USD)',
    'Hardware quantity', 'Training hardware',
    'Model accessibility', 'Training code accessibility',
    'Open model weights?', 'Citations', 'Confidence',
    'Notability criteria'
]

df_clean = df_clean[columnas_utiles].copy()
print(f"Regla 1 — Columnas seleccionadas: {df_clean.shape[1]} de {df.shape[1]}")

# REGLA 4 — Corrección de tipos
df_clean['Publication date'] = pd.to_datetime(df_clean['Publication date'], errors='coerce')
df_clean['Training dataset size (total)'] = pd.to_numeric(
    df_clean['Training dataset size (total)'], errors='coerce')
df_clean['Year'] = df_clean['Publication date'].dt.year
df_clean['Month'] = df_clean['Publication date'].dt.month
print(f"Regla 4 — Tipos corregidos y Year/Month extraídos")

# REGLA 3 — Eliminar duplicados por nombre de modelo
antes = df_clean.shape[0]
df_clean = df_clean.drop_duplicates(subset=['Model'], keep='first')
print(f"Regla 3 — Duplicados eliminados: {antes - df_clean.shape[0]}")

# REGLA 2 — Eliminar filas sin país, organización o fecha
antes = df_clean.shape[0]
df_clean = df_clean.dropna(subset=['Country (of organization)',
                                    'Organization',
                                    'Publication date'])
print(f"Regla 2 — Filas eliminadas por falta de país/org/fecha: {antes - df_clean.shape[0]}")

# REGLA 5 — Homologación de categorías
df_clean['Country (of organization)'] = (
    df_clean['Country (of organization)'].str.split(',').str[0].str.strip()
)
df_clean['Organization categorization'] = (
    df_clean['Organization categorization'].str.split(',').str[0].str.strip()
)
df_clean['Domain'] = (
    df_clean['Domain'].str.split(',').str[0].str.strip()
)
print(f"Regla 5 — Categorías homologadas")

# REGLA 6 — Imputación de categóricas con Unknown
categoricas_imputar = [
    'Organization categorization', 'Domain', 'Task',
    'Model accessibility', 'Training code accessibility',
    'Open model weights?', 'Confidence', 'Notability criteria'
]
for col in categoricas_imputar:
    df_clean[col] = df_clean[col].fillna('Unknown')
print(f"Regla 6 — Nulos en categóricas imputados con 'Unknown'")

# REGLA 7 — Nulos numéricos se mantienen
print(f"Regla 7 — Nulos en variables numéricas conservados")

# RESUMEN FINAL
print(f"\n{'='*60}")
print(f"RESUMEN DE LIMPIEZA")
print(f"{'='*60}")
print(f"Dataset original:      {df.shape[0]} filas | {df.shape[1]} columnas")
print(f"Dataset limpio:        {df_clean.shape[0]} filas | {df_clean.shape[1]} columnas")
print(f"Periodo cubierto:      {int(df_clean['Year'].min())} – {int(df_clean['Year'].max())}")
print(f"Países únicos:         {df_clean['Country (of organization)'].nunique()}")
print(f"Organizaciones únicas: {df_clean['Organization'].nunique()}")
print(f"Dominios únicos:       {df_clean['Domain'].nunique()}")

Regla 1 — Columnas seleccionadas: 20 de 57
Regla 4 — Tipos corregidos y Year/Month extraídos
Regla 3 — Duplicados eliminados: 6
Regla 2 — Filas eliminadas por falta de país/org/fecha: 98
Regla 5 — Categorías homologadas
Regla 6 — Nulos en categóricas imputados con 'Unknown'
Regla 7 — Nulos en variables numéricas conservados

RESUMEN DE LIMPIEZA
Dataset original:      3509 filas | 57 columnas
Dataset limpio:        3405 filas | 22 columnas
Periodo cubierto:      1950 – 2026
Países únicos:         45
Organizaciones únicas: 1306
Dominios únicos:       22
